# Week 02｜地震地図をつくる・改良する

授業の最後の30分に始め、このNotebookを宿題として仕上げます。最初は**CSVの一行につき、マーカーを一つ**。そこから自分の地図へ。


## 1｜CSVを用意する

授業サイトから**`eq.csv`**をダウンロードし、このNotebookと同じフォルダに保存します。2026年10月7日時点の地震データです。

もっと新しいデータを使うなら、[USGSのCSVダウンロードページ](https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php)で **Past Day → All Earthquakes** を選びます。ダウンロードしたCSVの名前に`csv_file`を変えてください。

最初の一度だけ、下のCellで地図の道具`folium`をインストールします。CSVを読む`csv`はPythonに最初から入っています。


In [1]:
%pip install folium

Note: you may need to restart the kernel to use updated packages.


## 2｜CSVを読む

`latitude`は緯度、`longitude`は経度、`mag`はマグニチュード。CSVの値は文字なので、地図に使う緯度・経度は`float()`で数に変えます。


In [2]:
import csv
import folium

csv_file = "eq.csv"

with open(csv_file, encoding="utf-8-sig") as file:
    earthquakes = list(csv.DictReader(file))

print(f"{len(earthquakes)}件のデータを読みました")

1850件のデータを読みました


## 3｜一行につき、マーカーを一つ

実行する前に、どこにどんな点が出るか予想しよう。`for`の中が各地震に対して繰り返されます。最後の`earthquake_map`でNotebookに地図を表示します。


In [3]:
earthquake_map = folium.Map(location=[20, 0], zoom_start=2)

for quake in earthquakes:
    folium.Marker(
        location=[float(quake["latitude"]), float(quake["longitude"])],
        tooltip=f"M {quake['mag']}"
    ).add_to(earthquake_map)

earthquake_map

## Challenge｜この地図を改良しよう

授業では一つ改良を始めます。宿題では**二つ以上の改良**を完成させよう。下のコードCellに、上の地図のコードをコピーして変更します。

| 改良の例 | ヒント |
| --- | --- |
| ベースマップを変える | `folium.Map(..., tiles="CartoDB positron")`。地図の見え方を比べる。 |
| ピンを円にする | `folium.Marker`を`folium.CircleMarker`に変え、`radius=5`、`fill=True`を追加する。 |
| 大きさによって色を変える | `float(quake["mag"])`と`if`で色を選び、円の`color`と`fill_color`に使う。空欄の`mag`は先に除く。 |
| 大きな地震だけ表示する | `if quake["mag"] and float(quake["mag"]) >= 4:`の中でマーカーを作る。 |
| 自分の工夫をする | 地域を絞る、円の大きさを変える、クリックで情報を見せる、など。 |

どの改良も、変更する前に結果を予想してから実行しよう。フィルターを使ったら、条件を満たす地震だけ表示されているか確認する。


In [4]:
# 上の地図のコードをコピーして、自分の改良を加えよう。
import folium
from branca.colormap import LinearColormap ##カラーマップをインポート

def draw_map(min_mag=4.0): ##darw_mapを作る
    earthquake_map = folium.Map(location=[40, 140], zoom_start=4) ##日本周辺が映るように

    colormap = LinearColormap(
        colors=["yellow", "orange", "darkred"], ##マグニチュードが低いと黄色、高くなるにつれて赤くなる（グラデーション）
        vmin=4,
        vmax=6,
        caption="マグニチュード", ##キャプションを付ける
    )

    sorted_quakes = sorted(earthquakes, key=lambda q: float(q["mag"])) ##マグニチュードが小さいものから表示する（大きいものが手前に表示されるようになる）

    for quake in sorted_quakes:
        mag = float(quake["mag"])
        if mag < min_mag: ##n以下のマグニチュードを表示しないように（今回は4以下）
            continue
        color = colormap(mag)
        folium.CircleMarker(
            location=[float(quake["latitude"]), float(quake["longitude"])],
            tooltip=f"M {quake['mag']}",
            radius=mag ** 3 / 10, ##マグニチュードが大きいと円の大きさも大きくなる
            color=color, ##円の外の色
            fill=True, ##円を塗りつぶす
            fill_opacity=0.7, ##円の透明度
        ).add_to(earthquake_map)

    colormap.add_to(earthquake_map)
    return earthquake_map

draw_map(4) ##()の中の数字以上のマグニチュードのみを表示

## 宿題｜自分の地震地図

授業で始めた地図を仕上げます。別のNotebookを作る必要はありません。

- タイトルと、誰に見せる地図かを書く。
- 二つ以上改良し、その理由と地図から気づいたことを書く。
- CSVの出典とダウンロード日を書く（付属CSVなら2026年10月7日時点のデータと明記）。
- 上から全Cellを実行し、地図が表示された状態で保存する。
- **完成した`.ipynb`と使用したCSVをUNIPAに提出する。**

**締切：2026年10月14日(水) 23:59 JST**


### 私の地図<br>学籍番号 [ID]<br>氏　　名 神郡　 正稀

**タイトル：**<br>
*日本周辺の地震情報*<br>

**誰に見せたい？：**<br>
*日本に住む人*<br>

**改良したこと・理由：**<br>
*改良1.最初に映る場所を日本周辺にした*<br>
*理由1.日本に住んでいるので日本の情報を真っ先に得たいと思ったから*<br>
*改良2.マグニチュードの大きさに応じて円の大きさと色の濃さが変わるようにした*<br>
*理由2.マグニチュードの大きさを直感で分かるようにしたかったから*

**地図から気づいたこと：** <br>
*日本周辺の地震の震源地は線状になっているように見える（プレートの形？）* <br>
*中国のほうは震源地がまばらに見える*<br>
*日本海側は太平洋側に比べて地震が少ない* <br>

**確かめたこと：**<br>
*Claudeに「こういうことをするためにはどんな関数を使えばいいか」を解説してもらった。*<br>
*コマンドを一つ一つどのように動作するか数字を変えたり文を消したりしながら実行して理解を深めた*<br>

**CSVの出典・ダウンロード日：**<br>
*https://earthquake.usgs.gov/earthquakes/feed/v1.0/csv.php*<br>
*2026年10月7日時点のUSGSデータ*
